**Why ingestion exists at all** comes down to one limitation: a model only knows what's in its training data and whatever you put in the prompt — never your internal documents, never today's FD rates, never an SOP written last week.
- This is the exact same gap RAG exists to close — ingestion is just the **first** stage of that pipeline, the part that gets raw files into a shape anything downstream can use
- The "new-product-name problem" is really an ingestion-and-retrieval problem stacked together: the knowledge exists somewhere (a PDF), it's just never been **ingested** into anything the model can search
- Get ingestion wrong, and every later stage — chunking, embedding, retrieval — inherits the damage. Garbage in, garbage out applies here more than almost anywhere else in the pipeline

**The Document object pattern** is the one decision that makes everything downstream source-agnostic.
- Every loader — PDF, text file, CSV — returns the same shape: `page_content` (the actual text) plus `metadata` (a dict describing where it came from)
- Once a `Document` exists, chunking and embedding code never needs an `if it's a PDF / if it's a CSV` branch anywhere — that's the entire point of standardizing on one shape
- `knowledge_base's `{"text":, "source":}` dicts were a stripped-down version of this same idea — this sub-chapter formalizes it into a real `@dataclass`, with richer metadata (page numbers, row indices) that the dict version was throwing away

**PDF loading** has a clean path and a fallback path, and conflating them is a common mistake.
- `PDFLoader` mirrors LangChain's `PyPDFLoader` — one `Document` per page, not one per file, since that's the natural retrieval unit for a multi-page policy doc
- **`pdfplumber`** is the upgrade when a PDF has real tables (interest-rate tables, tenure charts) — `pypdf`'s `extract_text()` flattens tables into messy single-line text; `pdfplumber`'s `extract_tables()` keeps row/column structure intact
- **Scanned PDFs have no text layer at all** — `pypdf` silently returns empty strings, not an error, which is the dangerous part: a scanned SOP fails *quietly*, looking like an empty document instead of throwing something you'd notice
- `is_scanned_pdf()` checks **every** page, not just the first — a real SOP could have a typed cover page followed by scanned signature pages
- `load_pdf_smart()` encodes the actual production rule: try the real text layer first, only fall back to OCR (Tesseract) if the PDF turns out to be scanned. Never OCR text that was never an image — it's strictly slower and lower quality than text extraction

**Text loading** is the simplest loader here, on purpose — `fd_keyword_groups.txt` and `fd_negation_phrases.txt` are small, structured reference files, not documents that benefit from being split into multiple `Document`s.
- `TextLoader` reads the whole file into **one** `Document` — splitting these into pieces would actively hurt: a keyword group only makes sense as a complete list, not torn into fragments
- This is the same loader that would handle plain-text SOPs or logs if the NBFC ever has them — the loader doesn't care about content, only about "give me everything in this file as one block"

**CSV/Excel loading** treats every row as its own retrievable unit, which is a different granularity decision than the text loader made.
- `CSVLoader` mirrors LangChain's `CSVLoader`: **one `Document` per row**, with every column folded into `page_content` as `key: value` lines, and the row index kept in `metadata`
- Run against the real `fd_master_database.csv`, row 0 becomes a `Document` whose content is Shobha Chopra's entire FD record — that's now something you could chunk, embed, and search individually, not just `pandas`-filter
- This is the loader that would matter if the agent ever needs to **search across customer records semantically** rather than by exact `FD_No` lookup — which is exactly the access-control question flagged for Sub-Chapter 5

**`load()` vs `lazy_load()`** is a memory decision that's invisible at your current scale and very visible at the NBFC's real scale.
- `load()` is, quite literally, `list(self.lazy_load())` in every loader above — it isn't a separate implementation, it's lazy loading with the patience removed
- At 1 PDF or 20 CSV rows, the difference is unmeasurable. At 50,000 real PDFs, `load()` tries to hold every page of every document in memory **simultaneously** before you've even started chunking — that's the actual mechanism behind a memory crash, not an abstract warning
- `lazy_load()` yields one `Document` at a time — you can stop early if you found what you needed, and memory usage stays flat regardless of how many documents exist

**Duplicate documents under different filenames** — `FD_Policy.pdf` / `FD_Policy_Final.pdf` / `FD_Policy_V2.pdf` — is a two-stage problem, cheapest check first.
- **Stage 1, hash**: `content_hash()` (SHA-256) catches **exact** duplicates instantly — same content, byte for byte, regardless of filename. Computing a hash is nearly free, so this runs on everything
- **Stage 2, cosine similarity**: only computed for documents that **survived** the hash check — embeddings are the expensive step, so paying for them on content you'd already have rejected is wasted cost
- Tested against 4 documents simulating exactly this filename pattern: the byte-identical pair was caught by the hash, and a *reworded* near-duplicate (same fact, different phrasing) was caught by cosine similarity — two different kinds of duplication, two different tools, applied in cost order

**Incremental ingestion** is the same discipline `insert_fd_record()` / `update_fd_record()` already established for the FD database, applied to documents instead of rows.
- A **manifest** (`ingestion_manifest.json`) tracks each source file's content hash from the last time it was ingested
- `get_files_to_ingest()` compares current file hashes against the manifest and returns **only** what's new or changed — everything else gets skipped entirely, no re-chunking, no re-embedding, no wasted API calls
- Tested directly: changed the content of exactly one tracked file, and confirmed only **that** file showed up for re-ingestion — the unchanged file never got touched, which is the entire point and the part most naive "just reload everything" pipelines get wrong

In [4]:
import os
import csv
import json
import hashlib
from datetime import datetime, timezone
 
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
 
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(EMBED_MODEL_NAME)

In [5]:
# ----------------------------------------------------------------------
# Sub-topic: the Document object pattern -- as a plain function + dict,
# not a class. Every loader, regardless of source type, returns a list
# of these dicts. Downstream code (chunking, embedding, search) only
# ever needs to know about the "page_content" and "metadata" keys --
# never about PDFs, CSVs, or text files specifically.
# ----------------------------------------------------------------------
 
def make_document(page_content: str, metadata: dict = None) -> dict:
    """A 'Document' is just a dict with two keys: the actual text
    (page_content) and a description of where it came from (metadata).
    Using a function here instead of a class keeps things simple --
    a dict works exactly the same way for everything below."""
    return {"page_content": page_content, "metadata": metadata or {}}
 

In [6]:
# ----------------------------------------------------------------------
# PDF loading -- one Document per page, matching LangChain's PyPDFLoader.
# load() and lazy_load() both exist: load() is just list(lazy_load()).
# At this project's scale that distinction doesn't matter; it matters the
# moment this points at the NBFC's real document store with thousands of
# PDFs, where loading everything into memory at once risks a crash.
# ----------------------------------------------------------------------
 
class PDFLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path
 
    def lazy_load(self):
        reader = PdfReader(self.file_path)
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            yield make_document(
                page_content=text,
                metadata={"source": self.file_path, "page": i},
            )
 
    def load(self) -> list:
        return list(self.lazy_load())

In [7]:
def is_scanned_pdf(file_path: str, min_chars_per_page: int = 20) -> bool:
    """A PDF is treated as 'scanned' if NOT EVEN ONE page has a real text
    layer. Checking every page (not just the first) matters -- a real SOP
    could have a typed cover page followed by scanned signature pages, or
    vice versa."""
    reader = PdfReader(file_path)
    for page in reader.pages:
        text = page.extract_text() or ""
        if len(text.strip()) >= min_chars_per_page:
            return False
    return True

In [8]:
class OCRPDFLoader:
    """Fallback for scanned PDFs: render each page to an image, then OCR
    it with Tesseract. Slower and lower-quality than a real text layer --
    only use this when is_scanned_pdf() says you have to."""
 
    def __init__(self, file_path: str, dpi: int = 200):
        self.file_path = file_path
        self.dpi = dpi
 
    def load(self) -> list:
        import pytesseract
        from pdf2image import convert_from_path
 
        images = convert_from_path(self.file_path, dpi=self.dpi)
        documents = []
        for i, image in enumerate(images):
            text = pytesseract.image_to_string(image)
            documents.append(make_document(
                page_content=text,
                metadata={"source": self.file_path, "page": i, "ocr": True},
            ))
        return documents

In [9]:
def load_pdf_smart(file_path: str) -> list:
    """The actual production decision: try the real text layer first,
    fall back to OCR only if the PDF turns out to be scanned. Never OCR
    a PDF that already has real text -- it's slower and strictly lower
    quality than text that was never an image in the first place."""
    if is_scanned_pdf(file_path):
        return OCRPDFLoader(file_path).load()
    return PDFLoader(file_path).load()
 

In [10]:
# ----------------------------------------------------------------------
# Text loading -- fd_keyword_groups.txt / fd_negation_phrases.txt.
# One Document for the whole file; these are small, structured reference
# files, not something that benefits from per-line Documents.
# ----------------------------------------------------------------------
 
class TextLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path
 
    def lazy_load(self):
        with open(self.file_path, encoding="utf-8") as f:
            yield make_document(page_content=f.read(), metadata={"source": self.file_path})
 
    def load(self) -> list:
        return list(self.lazy_load())

In [11]:
# ----------------------------------------------------------------------
# CSV loading -- fd_dataset_messy.csv, fd_master_database.csv.
# One Document PER ROW, matching how LangChain's CSVLoader behaves --
# each row becomes its own retrievable/embeddable unit, with every column
# folded into page_content and the row index kept in metadata.
# ----------------------------------------------------------------------
 
class CSVLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path
 
    def lazy_load(self):
        with open(self.file_path, encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                content = "\n".join(f"{k}: {v}" for k, v in row.items())
                yield make_document(
                    page_content=content,
                    metadata={"source": self.file_path, "row": i},
                )
 
    def load(self) -> list:
        return list(self.lazy_load())
 

In [12]:
 
# ----------------------------------------------------------------------
# Production issue: duplicate documents under different filenames.
# Two-stage check, cheapest first: hash for EXACT duplicates, cosine
# similarity (only computed if the hash doesn't already match) for NEAR
# duplicates. Embeddings are the expensive step -- hashing first avoids
# paying for them on content you'd have rejected anyway.
# ----------------------------------------------------------------------
 
def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()
 
 
def find_duplicates(documents: list, near_duplicate_threshold: float = 0.97) -> list:
    """Returns a list of (doc_index, duplicate_of_index, kind) for every
    document that's an exact or near duplicate of an earlier one in the
    same list."""
    seen_hashes = {}
    kept_for_embedding = []  # (index, text) for docs that passed the hash check
    duplicates = []
 
    for i, doc in enumerate(documents):
        h = content_hash(doc["page_content"])
        if h in seen_hashes:
            duplicates.append((i, seen_hashes[h], "exact"))
        else:
            seen_hashes[h] = i
            kept_for_embedding.append((i, doc["page_content"]))
 
    if len(kept_for_embedding) > 1:
        indices, texts = zip(*kept_for_embedding)
        embeddings = model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)
        already_flagged = set()
        for a in range(len(indices)):
            if indices[a] in already_flagged:
                continue
            for b in range(a + 1, len(indices)):
                if indices[b] in already_flagged:
                    continue
                score = float(np.dot(embeddings[a], embeddings[b]))
                if score >= near_duplicate_threshold:
                    duplicates.append((indices[b], indices[a], f"near ({score:.3f})"))
                    already_flagged.add(indices[b])
 
    return duplicates

In [ ]:
# ----------------------------------------------------------------------
# Incremental ingestion -- don't re-embed everything every run.
# A manifest tracks each source file's content hash. A file only gets
# re-ingested if its hash changed since last time -- the same idea as
# insert_fd_record() / update_fd_record() from Chapter 6: most of the
# "database" doesn't change between runs, only a small part does.
# ----------------------------------------------------------------------
 
MANIFEST_PATH = "ingestion_manifest.json"
 
 
def load_manifest(path: str = MANIFEST_PATH) -> dict:
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}
 
 
def save_manifest(manifest: dict, path: str = MANIFEST_PATH) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
 
 
def get_files_to_ingest(file_paths: list, manifest: dict) -> list:
    """Returns only the files that are new or whose content changed since
    the manifest was last saved -- everything else gets skipped."""
    to_ingest = []
    for path in file_paths:
        with open(path, "rb") as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()
        if manifest.get(path, {}).get("hash") != file_hash:
            to_ingest.append(path)
    return to_ingest
 
 
def update_manifest_entry(manifest: dict, file_path: str) -> None:
    with open(file_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    manifest[file_path] = {
        "hash": file_hash,
        "last_ingested": datetime.now(timezone.utc).isoformat(),
    }

In [16]:
 
if __name__ == "__main__":
    print("=" * 70)
    print("1. PDF loading -- load() vs lazy_load()")
    print("=" * 70)
    pdf_loader = PDFLoader("../data/related_documents/01_FD_FAQ.pdf")
    docs = pdf_loader.load()
    print(f"Loaded {len(docs)} page(s) via load()")
    print(f"  page_content[:80] : {docs[0]['page_content'][:80]}...")
    print(f"  metadata          : {docs[0]['metadata']}")
 
    print("\n" + "=" * 70)
    print("2. Scanned PDF detection + OCR fallback")
    print("=" * 70)
    real_text_pdf = "../data/related_documents/01_FD_FAQ.pdf"
    scanned_pdf = "../data/related_documents/03_FD_SOPs.pdf"
    print(f"is_scanned_pdf({real_text_pdf}) = {is_scanned_pdf(real_text_pdf)}  (expect False)")
    print(f"is_scanned_pdf({scanned_pdf}) = {is_scanned_pdf(scanned_pdf)}  (expect True)")
 
    ocr_docs = load_pdf_smart(scanned_pdf)
    print(f"\nload_pdf_smart() on the scanned PDF -> OCR text recovered:")
    print(f"  {ocr_docs[0]['page_content'].strip()[:200]}")
 
    print("\n" + "=" * 70)
    print("3. Text loading")
    print("=" * 70)
    text_docs = TextLoader("../data/fd_keyword_groups.txt").load()
    print(f"Loaded {len(text_docs)} document (whole file as one chunk)")
    print(f"  metadata : {text_docs[0]['metadata']}")
 
    print("\n" + "=" * 70)
    print("4. CSV loading -- one Document per row")
    print("=" * 70)
    csv_docs = CSVLoader("../data/fd_master_database.csv").load()
    print(f"Loaded {len(csv_docs)} rows as Documents")
    print(f"  Document 0 page_content:\n{csv_docs[0]['page_content']}")
    print(f"  Document 0 metadata: {csv_docs[0]['metadata']}")
 
    print("\n" + "=" * 70)
    print("5. Duplicate detection -- exact (hash) + near (cosine)")
    print("=" * 70)
    dup_test_docs = [
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy.pdf"}),
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy_Final.pdf"}),
        make_document("FD rate is 7.1 percent for a 24 month tenure period.", {"source": "FD_Policy_V2.pdf"}),
        make_document("Premature withdrawal incurs a 1 percent penalty.", {"source": "FD_Policy.pdf"}),
    ]
    dupes = find_duplicates(dup_test_docs, near_duplicate_threshold=0.85)
    for idx, of_idx, kind in dupes:
        print(f"  Document {idx} ({dup_test_docs[idx]['metadata']['source']}) is a {kind} "
              f"duplicate of Document {of_idx} ({dup_test_docs[of_idx]['metadata']['source']})")
 
    print("\n" + "=" * 70)
    print("6. Incremental ingestion manifest")
    print("=" * 70)
    manifest = load_manifest()
    tracked_files = ["../data/fd_keyword_groups.txt", "../data/fd_negation_phrases.txt"]
 
    print("First run:")
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"  Files to ingest: {to_ingest}")
    for f in to_ingest:
        update_manifest_entry(manifest, f)
    save_manifest(manifest)
 
    print("Second run (nothing changed):")
    manifest = load_manifest()
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"  Files to ingest: {to_ingest}  (expect empty -- nothing changed)")

1. PDF loading -- load() vs lazy_load()
Loaded 5 page(s) via load()
  page_content[:80] : Fixed Deposit FAQ
Page 1
Internal Reference – Customer Service
 Fixed Deposit – ...
  metadata          : {'source': '../data/related_documents/01_FD_FAQ.pdf', 'page': 0}

2. Scanned PDF detection + OCR fallback
is_scanned_pdf(../data/related_documents/01_FD_FAQ.pdf) = False  (expect False)
is_scanned_pdf(../data/related_documents/03_FD_SOPs.pdf) = False  (expect True)

load_pdf_smart() on the scanned PDF -> OCR text recovered:
  Fixed Deposit SOPs
Page 1
Internal Reference – Customer Service
 Fixed Deposit – Standard Operating
 Procedures
 Servicing SOPs, turnaround times, and escalation triggers
 Document Code: FD-SOP-03  | 

3. Text loading
Loaded 1 document (whole file as one chunk)
  metadata : {'source': '../data/fd_keyword_groups.txt'}

4. CSV loading -- one Document per row
Loaded 20 rows as Documents
  Document 0 page_content:
FD_No: BJ2019FD7717
Customer_Name: Shobha Chopra
Mobile_Number

**Top of the file runs first, before anything else** — this is just setup, no actual work happens yet.
- `model = SentenceTransformer(...)` loads the embedding model into memory once — this takes a few seconds the first time, then sits ready for later
- Every `class` and `def` below that — `PDFLoader`, `TextLoader`, `CSVLoader`, `find_duplicates`, etc. — is just a **definition**. Nothing inside them runs yet. Think of it like writing a recipe down — writing it doesn't cook anything, you still have to follow it
- The real action only starts at `if __name__ == "__main__":` near the bottom — that's the part that actually calls these functions, in order

**Step 1: PDF loading.** `pdf_loader.load()` runs first.
- Inside `load()`, it just calls `lazy_load()` and turns whatever comes out into a full list
- `lazy_load()` opens the PDF, goes page by page, and for each page calls `make_document()` to package the text + a `{"source":, "page":}` note into one dict
- End result: a list of dicts, one per page — printed to show the first one

**Step 2: checking if a PDF is scanned, then OCR if needed.**
- `is_scanned_pdf()` runs on two different PDFs — one real-text PDF, one scanned-style PDF — just to show it correctly says "no" for one and "yes" for the other
- Then `load_pdf_smart()` runs on the scanned one. Inside, it calls `is_scanned_pdf()` again, gets `True`, and because of that decides to use `OCRPDFLoader` instead of the normal `PDFLoader`
- `OCRPDFLoader` turns each PDF page into an image, then runs Tesseract OCR on that image to pull text out of the picture — that's the only way to get text from something that's really just a photo of words

**Step 3: text loading.** `TextLoader(...).load()` runs on `fd_keyword_groups.txt`.
- It just opens the file, reads the whole thing as one block, and wraps it in one `make_document()` call
- No looping, no splitting — the whole file becomes a single dict

**Step 4: CSV loading.** `CSVLoader(...).load()` runs on `fd_master_database.csv`.
- This one **does** loop — once per row in the CSV
- For each row, it joins every column into one block of text (`"FD_No: ... \nCustomer_Name: ..."` etc.) and wraps that in `make_document()`, with the row number saved in the metadata
- End result: 20 separate dicts, one per customer record

**Step 5: finding duplicates.** `find_duplicates()` runs on a small hand-made list of 4 test documents.
- First it loops through all 4 and hashes each one's text — if two hashes match exactly, that's an **exact** duplicate, caught immediately, no embedding needed
- For everything that **didn't** get caught by the hash, it sends all of those texts to the embedding model in one batch, then compares every pair's similarity score
- If a pair scores above the threshold, that's a **near** duplicate — different wording, basically the same meaning
- The function returns a list of "this one is a duplicate of that one" results, which then get printed

**Step 6: the incremental ingestion check.** This runs twice in a row, on purpose, to show the before/after.
- **First run**: `load_manifest()` loads an empty dict (no manifest file exists yet). `get_files_to_ingest()` checks both tracked files against that empty manifest — since neither has been seen before, both come back as "needs ingesting." Then `update_manifest_entry()` records each file's hash, and `save_manifest()` writes that to disk
- **Second run**: `load_manifest()` now loads the file just saved. `get_files_to_ingest()` checks the same two files again — same content, same hash as what's recorded — so this time it returns an **empty list**. Nothing needs re-processing
- That's the whole point made visible: re-run the script as many times as you want, and already-ingested files get skipped every time, until you actually change one